In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Loss
from scipy.optimize import minimize
from tensorflow.keras.initializers import GlorotUniform, GlorotNormal
from joblib import Parallel, delayed
import time
import keras.backend as K

In [ ]:
# Define DPD custom loss function as a class
class DPDLoss(Loss):
    def __init__(self, sigma, alpha):
        super().__init__()
        self.alpha = alpha
        self.sigma = float(sigma)

    def call(self, y_true, y_pred):
        diff = (y_true - y_pred)/self.sigma
        diff_sq = tf.clip_by_value(diff**2, 1e-10, 1e6)  # Prevents extreme values
        normal_dist = tf.exp(-0.5 * diff_sq)
        normal_dist = tf.clip_by_value(normal_dist, 1e-10, 1.0)  # Keep in safe range
        loss = 1/((self.sigma**self.alpha)*((1+self.alpha)**0.5)) - (1+1/self.alpha)*tf.reduce_mean((normal_dist/self.sigma)**self.alpha)
        return loss

class HuberLoss(Loss):
    def __init__(self, delta=1.345, sig=None):
        super().__init__()
        self.delta = delta
        self.sig = sig

    def call(self, y_true, y_pred):
        error = (y_true - y_pred)/(self.sig + K.epsilon())
        # Huber loss computation
        is_small_error = tf.abs(error) <= self.delta
        squared_loss = 0.5 * tf.square(error)
        linear_loss = self.delta * (tf.abs(error) - 0.5 * self.delta)
        loss = tf.where(is_small_error, squared_loss, linear_loss)
        return tf.reduce_mean(loss)

class TukeyLoss(Loss):
    def __init__(self, delta=4.685, sig=None):
        super().__init__()
        self.delta = delta
        self.sig = sig

    def call(self, y_true, y_pred):
        error = (y_true - y_pred)/(self.sig + K.epsilon())
        # Apply Tukey's biweight function
        mask = tf.abs(error) <= self.delta
        loss = tf.where(mask, (self.delta**2 / 6) * (1 - (1 - (error / self.delta) ** 2) ** 3), self.delta**2 / 6)
        return tf.reduce_mean(loss)  # Return mean loss over batch

class LMLSLoss(Loss):
    def __init__(self):
        super().__init__()

    def call(self, y_true, y_pred):
        r = y_pred - y_true  # Compute residual error
        loss = tf.math.log(1 + (r**2) / 2)  # Apply LMLS formula
        return tf.reduce_mean(loss)  # Take mean loss over batch

class LTSLoss(Loss):
    def __init__(self, h):
        super().__init__()
        self.h = h

    def call(self, y_true, y_pred):
        residuals = tf.square(y_true - y_pred)  # Compute squared residuals
        sorted_residuals = tf.sort(residuals)   # Sort in ascending order
        #num_elements = tf.size(residuals)
        #k = tf.minimum(self.h, num_elements)
        k = self.h
        trimmed_residuals = sorted_residuals[:k]  # Keep only k smallest residuals
        return tf.reduce_mean(trimmed_residuals)  # Compute mean of the kept residuals

class LTALoss(Loss):
    def __init__(self, h):
        super().__init__()
        self.h = h

    def call(self, y_true, y_pred):
        residuals = tf.abs(y_true - y_pred)  # Compute absolute residuals
        sorted_residuals = tf.sort(residuals)   # Sort in ascending order
        #num_elements = tf.size(residuals)
        #k = tf.minimum(self.h, num_elements)
        k = self.h
        trimmed_residuals = sorted_residuals[:k]  # Keep only k smallest residuals
        return tf.reduce_mean(trimmed_residuals)  # Compute mean of the kept residuals

def trimmed_mean(arr, trim_fraction):
    arr_sorted = np.sort(arr)  # Step 1: Sort array
    trim_count = int(len(arr) * trim_fraction)  # Step 2: Compute number of elements to trim
    
    if trim_count == 0:  # Ensure at least one element is trimmed if possible
        return np.mean(arr)
    
    trimmed_arr = arr_sorted[:-trim_count]  # Step 3: Remove largest 20%
    return np.mean(trimmed_arr)  # Step 4: Compute mean of remaining elements

def H(sigma, alpha, diff):
    normal_dist = np.exp(-0.5 * (diff / sigma) ** 2) / sigma
    loss = 1 / ((sigma ** alpha) * np.sqrt(1 + alpha)) - (1 + 1 / alpha) * np.mean(normal_dist ** alpha)
    return loss

def sig_hat_MAD(resi):
    return 1.4826 * np.median(np.abs(resi - np.median(resi)))

In [ ]:
def g(x):
    m1 = np.array([0,0.75]); m2 = np.array([0.5,-0.5]); m3 = np.array([-0.75,0])
    return np.exp(-np.sum((x-m1)**2)) + np.exp(-2*np.sum((x-m2)**2)) + np.exp(-3*np.sum((x-m3)**2))

## MLE, Huber, Tukey, LTA, LTS, LMLS

In [ ]:
## MLE, Huber, Tukey, LTA, LTS, LMLS
n = 200
s_g = 0.05

def TMSE_MLE_Huber(r):
    np.random.seed(r)
    tf.random.set_seed(r)

    ## Data generation
    X = np.random.uniform(-1,1,(n,2)); gi = [g(X[i]) for i in range(n)]; e = np.random.normal(0, s_g, n)
    y = gi + e

    # Inserting contaminations
    n1 = int(n*delta)
    X_cont = np.random.uniform(-10,10,(n1,2)); y_cont = np.random.normal(10, np.sqrt(10), n1)
    r_ind = np.random.choice(range(n), size=n1, replace=False)
    X[r_ind] = X_cont; y[r_ind] = y_cont
    Y = y.reshape(-1,1)

    ## Test data generation
    X_test = np.random.uniform(-1,1,(n,2)); gi_test = [g(X_test[i]) for i in range(n)]; e_test = np.random.normal(0, s_g, n)
    y_test = gi_test + e_test
    Y_test = y_test.reshape(-1,1)

    n_epochs = 3000
    #### MLE / Usual Backpropagation
    # Define model
    model = keras.Sequential([
        keras.layers.Dense(30, activation='gelu', kernel_initializer=GlorotUniform(seed=42)),
        keras.layers.Dense(1, kernel_initializer=GlorotUniform(seed=42))
    ])
    # Model fitting MLE
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    model.fit(X, Y, epochs=n_epochs, verbose = 0)
    # Train data APE MLE
    mle_pred = model.predict(X, verbose=0).flatten()
    tmse = trimmed_mean((y - mle_pred) ** 2, trim_fraction=delta)
    # Test data APE MLE
    mle_pred_test = model.predict(X_test, verbose=0).flatten()
    mse_test = np.mean((y_test - mle_pred_test) ** 2)
    
    ## scale related estimates used for other methods
    diff = y-mle_pred; sig0 = sig_hat_MAD(diff)
    abs_diff = np.abs(diff); h = sum(abs_diff < 3*sig_hat_MAD(abs_diff))
    
    #### Huber
    # Define model
    model = keras.Sequential([
        keras.layers.Dense(30, activation='gelu', kernel_initializer=GlorotUniform(seed=42)),
        keras.layers.Dense(1, kernel_initializer=GlorotUniform(seed=42))
    ])
    # Model fitting Huber
    model.compile(optimizer=Adam(learning_rate=0.001), loss=HuberLoss(delta = 1.345, sig = sig0))
    model.fit(X, Y, epochs=n_epochs, verbose = 0)
    # Train data APE Huber
    huber_pred = model.predict(X, verbose=0).flatten()
    tmse_huber = trimmed_mean((y - huber_pred) ** 2, trim_fraction=delta)
    # Test data APE MLE
    huber_pred_test = model.predict(X_test, verbose=0).flatten()
    mse_test_huber = np.mean((y_test - huber_pred_test) ** 2)

    #### Tukey
    # Define model
    model = keras.Sequential([
        keras.layers.Dense(30, activation='gelu', kernel_initializer=GlorotUniform(seed=42)),
        keras.layers.Dense(1, kernel_initializer=GlorotUniform(seed=42))
    ])
    # Model fitting Tukey
    model.compile(optimizer=Adam(learning_rate=0.001), loss=TukeyLoss(delta = 4.685, sig = sig0))
    model.fit(X, Y, epochs=n_epochs, verbose = 0)
    # Train data APE Tukey
    tukey_pred = model.predict(X, verbose=0).flatten()
    tmse_tukey = trimmed_mean((y - tukey_pred) ** 2, trim_fraction=delta)
    # Test data APE MLE
    tukey_pred_test = model.predict(X_test, verbose=0).flatten()
    mse_test_tukey = np.mean((y_test - tukey_pred_test) ** 2)
    
    #### LTA
    # Define model
    model = keras.Sequential([
        keras.layers.Dense(30, activation='gelu', kernel_initializer=GlorotUniform(seed=42)),
        keras.layers.Dense(1, kernel_initializer=GlorotUniform(seed=42))
    ])
    # Model fitting
    model.compile(optimizer=Adam(learning_rate=0.001), loss=LTALoss(h=h))
    model.fit(X, Y, epochs=n_epochs, verbose = 0)
    # Train data APE Huber
    lta_pred = model.predict(X, verbose=0).flatten()
    tmse_lta = trimmed_mean((y - lta_pred) ** 2, trim_fraction=delta)
    # Test data APE MLE
    lta_pred_test = model.predict(X_test, verbose=0).flatten()
    mse_test_lta = np.mean((y_test - lta_pred_test) ** 2)
    
    #### LTS
    # Define model
    model = keras.Sequential([
        keras.layers.Dense(30, activation='gelu', kernel_initializer=GlorotUniform(seed=42)),
        keras.layers.Dense(1, kernel_initializer=GlorotUniform(seed=42))
    ])
    # Model fitting
    model.compile(optimizer=Adam(learning_rate=0.001), loss=LTSLoss(h=h))
    model.fit(X, Y, epochs=n_epochs, verbose = 0)
    # Train data APE Huber
    lts_pred = model.predict(X, verbose=0).flatten()
    tmse_lts = trimmed_mean((y - lts_pred) ** 2, trim_fraction=delta)
    # Test data APE MLE
    lts_pred_test = model.predict(X_test, verbose=0).flatten()
    mse_test_lts = np.mean((y_test - lts_pred_test) ** 2)

    #### LMLS
    # Define model
    model = keras.Sequential([
        keras.layers.Dense(30, activation='gelu', kernel_initializer=GlorotUniform(seed=42)),
        keras.layers.Dense(1, kernel_initializer=GlorotUniform(seed=42))
    ])
    # Model fitting
    model.compile(optimizer=Adam(learning_rate=0.001), loss=LMLSLoss())
    model.fit(X, Y, epochs=n_epochs, verbose = 0)
    # Train data APE Huber
    lmls_pred = model.predict(X, verbose=0).flatten()
    tmse_lmls = trimmed_mean((y - lmls_pred) ** 2, trim_fraction=delta)
    # Test data APE MLE
    lmls_pred_test = model.predict(X_test, verbose=0).flatten()
    mse_test_lmls = np.mean((y_test - lmls_pred_test) ** 2)

    return [tmse, mse_test, tmse_huber, mse_test_huber, tmse_tukey, mse_test_tukey, tmse_lta, mse_test_lta, tmse_lts, mse_test_lts, tmse_lmls, mse_test_lmls, sig0, h]

In [ ]:
#TMSE_MLE_Huber(9)

In [ ]:
delta = 0     # Contamination proportion
print('cont prop:', delta)

start = time.time()
result_list_mle = Parallel(n_jobs = 104)(delayed(TMSE_MLE_Huber)(r) for r in range(1000))
end = time.time()
TAPE_mle = np.array(result_list_mle)
print('Result shape:', TAPE_mle.shape)
print('time taken:', end-start)
df1 = pd.DataFrame(np.mean(TAPE_mle, axis = 0)[:-2].reshape(-1,2),columns=['train','test'], index=['MLE','Huber','Tukey','LTA','LTS','LMLS'])
df1 # Answer reported in paper

In [ ]:
df = pd.DataFrame(TAPE_mle)
aa = ['tmse', 'mse_test', 'tmse_huber', 'mse_test_huber', 'tmse_tukey', 'mse_test_tukey', 'tmse_lta', 'mse_test_lta', 'tmse_lts', 'mse_test_lts', 'tmse_lmls', 'mse_test_lmls', 'sig0', 'h']
df.to_csv("TAPE_comp_0.csv", index=False, header=aa)
df1.to_csv("TAPE_comp_result_0.csv")

# DPD

In [ ]:
## DPD
n = 200
s_g = 0.05
#delta = 0.3     # Contamination proportion
# Specify correct initial sigma in the following function

al = np.array([0.1,0.3,0.5,0.7,1])

def TMSE(r, alpha):
    np.random.seed(r)
    tf.random.set_seed(r)

    ## Data generation
    X = np.random.uniform(-1,1,(n,2)); gi = [g(X[i]) for i in range(n)]; e = np.random.normal(0, s_g, n)
    y = gi + e

    # Inserting contaminations
    n1 = int(n*delta)
    X_cont = np.random.uniform(-10,10,(n1,2)); y_cont = np.random.normal(10, np.sqrt(10), n1)
    r_ind = np.random.choice(range(n), size=n1, replace=False)
    X[r_ind] = X_cont; y[r_ind] = y_cont
    Y = y.reshape(-1,1)

    ## Test data generation
    X_test = np.random.uniform(-1,1,(n,2)); gi_test = [g(X_test[i]) for i in range(n)]; e_test = np.random.normal(0, s_g, n)
    y_test = gi_test + e_test
    Y_test = y_test.reshape(-1,1)

    # Define the model
    model = keras.Sequential([
        keras.layers.Dense(30, activation='gelu', kernel_initializer=GlorotUniform(seed=42)),
        keras.layers.Dense(1, kernel_initializer=GlorotUniform(seed=42))
    ])
    
    sig0 = 0.1  # Give initial value of sigma
    num_iterations = 15
    for i in range(num_iterations):
        model.compile(optimizer=Adam(learning_rate=0.001), loss=DPDLoss(sigma = sig0, alpha=alpha))
        if(i>0):
            model.set_weights(weights)
        model.fit(X, Y, epochs = 200, verbose = 0)
        dpd_pred = model.predict(X, verbose=0)
        #print(dpd_pred.sum())
        weights = model.get_weights()
        #print(weights)
        diff = (Y - dpd_pred).flatten()
        sig0 = minimize(H, sig0, args = (alpha,diff), method='L-BFGS-B', bounds = [(0.001,10)]).x[0]
        #print(i, sig0)
    
    # Train data APE
    tmse = trimmed_mean(diff ** 2, trim_fraction=delta)
    # Test data APE
    dpd_pred_test = model.predict(X_test, verbose=0).flatten()
    mse_test = np.mean((y_test - dpd_pred_test) ** 2)
    
    return [tmse, mse_test]

def TMSE_arr(r):
    return np.array([TMSE(r, alpha) for alpha in al])

In [ ]:
#TMSE_arr(0)

In [ ]:
delta = 0
print('cont prop:', delta)
start = time.time()
result_list = Parallel(n_jobs = -1)(delayed(TMSE_arr)(r) for r in range(1000))
end = time.time()
TAPE = np.array(result_list)
print(TAPE.shape)
print('time taken:', end-start)
APE = np.mean(TAPE, axis = 0)
print(APE)

df = pd.DataFrame(APE)
sqTAPE = TAPE**0.5
sqAPE = np.mean(sqTAPE, axis = 0)
df1 = pd.DataFrame(sqAPE)
df.to_csv("APE_0.csv", index=False, header=['train', 'test'])
df1.to_csv("sqAPE_0.csv", index=False, header=['train', 'test'])

In [ ]:
delta = 0.1
print('cont prop:', delta)
start = time.time()
result_list = Parallel(n_jobs=-1)(delayed(TMSE_arr)(r) for r in range(1000))
end = time.time()
TAPE = np.array(result_list)
print(TAPE.shape)
print('time taken:', end-start)
APE = np.mean(TAPE, axis = 0)
print(APE) # Answer reported in paper. 1st column is for Avg TMSE (on train data), 2nd Column is for Avg MSE (on test data)

In [ ]:
df = pd.DataFrame(APE)
sqTAPE = TAPE**0.5
sqAPE = np.mean(sqTAPE, axis = 0)
df1 = pd.DataFrame(sqAPE)
df.to_csv("APE_10.csv", index=False, header=['train', 'test'])
df1.to_csv("sqAPE_10.csv", index=False, header=['train', 'test'])